# 🫀 퀘스트 46 · Q3-B — **사전확률 셔플 대조**(Q3 의 C4 를 고친다)

| | **MedKOS / `notebooks/quest46_q3b_prior_shuffle.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` — 층② 점수 눈금 |
| 부모 런 | `quest46_q3_prior_em` (공식 실행 `20260804T1226`) |
| 성격 | **대조 수리 런** — 새 가설이 아니라 **자를 고친다**(R35 ①) |

## 왜 이 런이 필요한가

Q3 은 이렇게 끝났다.

```
C0 ✅  C1 ✅   max|Δ| = 0.0e+00 (구성 항등 · 구현은 정확)
C2 ✅  오라클 이득 +0.2151 [+0.0852, +0.2994] · MDE 0.1071
       전역 raw 0.3362 → oracle 0.5514  (매크로 0.5389 수준까지 올라온다)
C3 ⚠️  EM 절대 Δ −0.0185 [−0.1014, +0.0773] · 회복률 −0.086
C4 ⚠️  라벨치환 영점 이득 oracle +0.2613 [+0.2475, +0.2734] > 관측 +0.2151
```

C4 가 「관측이 영점 안」이라고 경고했고 요약은 C2 ✅ 를 의심하라고 찍었다.
**그런데 그 대조가 잘못 짜였다.**

```
관측   raw 0.3362  →  oracle 0.5514      이득 +0.2151
영점   raw ≈0.106  →  oracle ≈0.367      이득 +0.2613
              └─ ★ 출발점이 다르다. 이득끼리 비교할 수 없다
```

라벨을 치환하면 기저가 무력해져 **raw 가 훨씬 낮은 데서 출발**한다. 낮은 데서 출발하면
같은 처치로도 Δ 가 크게 나온다(천장 효과). 헤드룸으로 정규화하면 관측 0.324 > 영점 0.292 로
**부호가 뒤집힌다**. **판정과 대조를 다른 통계량 위에 두면 어긋난다**(R40 ②)를 C4 에서 반복했다.

⚠️ 그리고 Q3 은 **영점의 raw 절대 수준을 로그에 안 남겼다**(이득만 찍었다). 위 0.106 은
무정보 랭커의 AP ≈ 유병률이라는 **이론값 추정**이지 실측이 아니다. 이 런이 실측한다.

## 무엇을 바꾸나 — **사전확률 셔플 대조**

라벨을 치환하는 대신 **π\* 값들을 레코드끼리 셔플**한다(derangement · 자기 값을 받는 레코드 없음).

| | 기저 점수 | raw | 주입되는 시프트 값 집합 | 레코드↔사전확률 대응 |
|---|---|---|---|---|
| `oracle` | 관측 | **0.3362** | {π\*} | **맞다** |
| `shuffled` | 관측 | **0.3362 (구성으로 동일)** | {π\*} (같은 집합) | **깨졌다** |

→ 두 팔의 차이는 **오직 대응**뿐이다. 「유병률 주입」과 「자기 사전확률 정렬」이 정확히 갈린다.

## 관문 (사전등록)

★ **읽는 순서가 판정의 일부다.** B1 이 주 관문이고, B1 이 무너지면 전역 PR-AUC 위의
이득 해석은 **C2 를 포함해 전부** 폐기한다.

| 관문 | 무엇 | 통과 기준 |
|---|---|---|
| **B0** | 재현·항등 — Q3 의 raw·매크로를 다시 낸다 + 셔플 팔의 raw 가 오라클 팔과 **정확히 같다** | `max\|Δraw\| = 0`, 구성 증명. 재현 \|Δ\| ≤ 0.005 |
| **B1 ★★★ 주 관문** | `oracle − shuffled` **짝지은 차**(같은 부트스트랩 재표집) | CI 가 0 을 뗄 것. ❌/미결이면 **전역 PR-AUC 위의 이득 해석을 폐기** |
| **B2 ★★** | 라벨치환 영점을 **Δ 가 아니라 절대 수준**으로 비교(C4 의 결함 교정) | 관측 oracle 수준이 영점 oracle 수준을 넘을 것 |
| **B3** | EM 에도 같은 셔플 대조 (`em − em_shuffled`) | 참고. EM 이 이미 음수라 여기서 결론이 나지 않을 수 있다 |
| **B4 ★★** | `π̂` 붕괴 진단 — clip 바닥에 붙은 레코드와 그 **pooled 기여** | 관문 아님. Q3 의 실패 기전을 특정한다 |
| **B5** | 대안 추정기(BBSE · 평균사후) vs EM — **`\|π̂−π*\|`** 로 비교 | 관문 아님. 추정 지표라 pooled 오염과 무관하다 |
| **B6** | 결론 검산표 | R38 ⑦ · R39 ⑤ |

### 판정표

- **B1 ✅ · B2 ✅** → 오라클 이득은 **자기 사전확률 정렬**에서 온다. C2 ✅ 가 **살아난다**.
  방법 A 의 병목은 **추정**이고, B5 가 다음 추정기를 지목한다
- **B1 ❌** → 이득이 **유병률 주입 자체**에서 온다. 전역 PR-AUC 는 이 질문의 지표로
  **못 쓴다** → Q4 의 사전등록 기준(「전역 PR-AUC > 0」)도 **같이 무효**다. 지표부터 새로 세운다
- **B1 ✅ · B2 ❌** → 대응은 중요한데 절대 수준이 영점을 못 넘는다 → 코호트·지배 지분 문제

⚠️ **MLLS 는 별도 팔이 아니다** — Saerens EM 이 곧 MLLS(maximum likelihood label shift)다.
Q3 인수인계에 「BBSE·MLLS」로 적혀 있지만 **실제로 다른 건 BBSE 뿐**이라 그것만 넣는다.

⚠️ **새 데이터 0** — `svdb_data5.npz` 만. 분할·기저·보정은 Q3 과 **같은 결정 절차**를 밟는다.


In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def mde(lo, hi):
    return (hi - lo) / 2.0 if np.isfinite(lo) and np.isfinite(hi) else float("nan")

def boot_mean(v, seed, nb=3000, q=2.5):
    d = np.asarray(v, float); d = d[np.isfinite(d)]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), len(d)
    rng = np.random.RandomState(seed)
    b = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)]
    return (float(d.mean()), float(np.percentile(b, q)),
            float(np.percentile(b, 100 - q)), len(d))

def spearman(a, b):
    ra = np.asarray(a, float).argsort().argsort().astype(float)
    rb = np.asarray(b, float).argsort().argsort().astype(float)
    if np.std(ra) < 1e-12 or np.std(rb) < 1e-12:
        return float("nan")
    return float(np.corrcoef(ra, rb)[0, 1])

def need_super(n, half, eff, p80=False):
    """★ 우월 프레임. 효과가 0 근처면 **해석 불가**다 — 호출부에서 그렇게 찍는다(R37 ① · R41 ②)."""
    if not np.isfinite(half) or abs(eff) < 1e-9 or n < 1:
        return float("nan")
    r = float(n) * (half / abs(eff)) ** 2
    return r * 2.04 if p80 else r

def ece_of(p, y, nbin=15):
    p = np.asarray(p, float); y = np.asarray(y, float)
    edges = np.linspace(0.0, 1.0, nbin + 1)
    e = 0.0
    for i in range(nbin):
        m = (p >= edges[i]) & (p < edges[i + 1] if i < nbin - 1 else p <= edges[i + 1])
        if not m.any():
            continue
        e += (m.mean()) * abs(p[m].mean() - y[m].mean())
    return float(e)

def derangement(n, rng):
    """★ 자기 자리를 받는 원소가 **하나도 없는** 순열. 셔플 대조가 「일부는 제 값」이면
    대조가 희석된다 — 그래서 완전 교란을 강제한다."""
    for _ in range(1000):
        p = rng.permutation(n)
        if not np.any(p == np.arange(n)):
            return p
    p = np.roll(np.arange(n), 1)     # 최후 수단: 순환 이동도 완전 교란이다
    return p

class AssetError(RuntimeError): pass
print("CELL 0 ✅")


In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, importlib, time, warnings
importlib.invalidate_caches(); warnings.filterwarnings("ignore")

SMOKE = os.environ.get("MEDKOS_SMOKE") == "1"
_ENV_ROOT = os.environ.get("MEDKOS_DRIVE_ROOT")
if _ENV_ROOT:
    DRIVE_ROOT = _ENV_ROOT
else:
    try:
        from google.colab import drive; drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = "/content/drive/MyDrive"
    except Exception as e:
        print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0, IDX_S = 20260804, 1
FULL_K = tuple(range(4, 33))
RHY_K  = (5, 10, 20, 32)
MIN_S, MIN_N = 25, 25

# ── 비용 손잡이(스모크에서만 축소) — 관문 문턱은 절대 안 건드린다
NB_BOOT = 200 if SMOKE else 1000
N_PERM  = 3   if SMOKE else 50
N_SHUF  = 3   if SMOKE else 10     # 사전확률 셔플(derangement) 개수

# ── ★★★ 사전등록: 읽는 순서. B1 이 무너지면 전역 위의 이득 해석을 **전부** 폐기한다
READ_ORDER = ("B0", "B1", "B2", "B3", "B4", "B5", "B6")
GATE_DEP = {"B1": ["B0"], "B2": ["B0"], "B3": ["B0", "B1"]}
PRIMARY = "oracle_minus_shuffled"   # ★ 주 통계량은 **짝지은 차**다(비가 아니다 · R40 ②)
MACRO_ROLE = "identity_control"     # 매크로는 여기서도 판정 지표가 아니다

ARM_MONOTONE = {"raw": True, "identity": True, "oracle": True,
                "shuffled": True, "em": True, "em_shuffled": True, "bbse": True}
TOL_IDENT = 1e-12
TOL_REPRO = 0.005                   # Q3 재현 허용오차(로그가 4자리라 이보다 못 좁힌다)
SPLIT_PATTERN = ("TRAIN", "TEST", "DEV", "TRAIN", "TEST", "TRAIN", "DEV", "TEST",
                 "TRAIN", "TEST", "DEV", "TRAIN", "TEST", "TRAIN", "DEV", "TEST",
                 "TRAIN", "TEST", "DEV", "TRAIN")

SV5 = os.path.join(MITBIH, "svdb_data5.npz")

# ── Q3 공식 실행(20260804T1226) 실측 — 재현 앵커
REF = dict(
    raw=0.3362, oracle=0.5514, em=0.3177,
    macro_prauc=0.538912, macro_auroc=0.871891,
    pi_tr=0.08074, n_test=20, n_dev=14, n_train=22, dominant=0.344,
    ece_dev=0.01511, ece_test=0.05458,
    em_hp=(100, 1e-9, 0.01), dev_pi_err=0.03588,
    c2_gain=0.2151, c2_lo=0.0852, c2_hi=0.2994, c2_mde=0.1071,
    c3_gain=-0.0185, c3_lo=-0.1014, c3_hi=0.0773, c3_recovery=-0.086,
    null_or=0.2613, null_or_lo=0.2475, null_or_hi=0.2734,
    null_em=0.0163, rho_pi=0.3880, pi_err_mean=0.06842,
    pi_err_med=0.02376, pi_err_max=0.56641)

RULE_CHECK = {
    "R11 / R11-b":       "전역이 주 지표인 예외 런 — 지배 지분·제외·GMIN_S 병기 · 단독 인용 금지",
    "R16 fallback 없음": "자산 없으면 **중단**",
    "R22 누출 없음":     "기저 보정·EM 하이퍼를 **DEV 에서만** 고정",
    "R26 / R38 ②":      "영점은 **측정**한다 — 0 을 가정하지 않는다",
    "R29 ② 분기 금지":   "★★ B0 이 깨지면 아래를 **안 읽는다**",
    "R33 ① MDE":         "관문마다 MDE 를 내고 점추정과 비교. **미결 ≠ 등가**",
    "R34 ③ 대조 보장":   "★★★ 셔플 팔의 raw 는 오라클 팔과 **구성으로 동일**하다",
    "R35 ① 자를 먼저":   "★★★ 이 런은 가설이 아니라 **자(대조)를 고친다**",
    "R38 ⑦ 요약 정합":   "★ 영점의 **절대 수준**을 로그에 남긴다 — Q3 은 이득만 찍어 비교 불가였다",
    "R40 ② 같은 통계량": "★★★ 판정과 대조를 **같은 통계량·같은 출발점** 위에 둔다(C4 가 어긴 것)",
    "R41 ② 0 근처":      "효과가 0 근처면 필요표본은 **해석 불가**",
}

CONFIG = dict(
    exp="quest46_q3b_prior_shuffle", quest="ailab-2026-0046", step="prior-shuffle-control",
    parent_exp=["quest46_q3_prior_em"],
    purpose=("**대조 수리 런.** Q3 의 C4 는 라벨치환 영점을 썼는데, 그러면 기저가 무력해져 "
             "**raw 가 관측보다 낮은 데서 출발**한다(관측 0.3362 vs 영점 ≈0.106). 낮은 데서 "
             "출발하면 같은 처치로도 Δ 가 크게 나오므로(천장 효과) **이득끼리 비교할 수 없다** — "
             "헤드룸으로 정규화하면 부호가 뒤집힌다(관측 0.324 > 영점 0.292). "
             "즉 C4 ⚠️ 는 「C2 가 가짜다」가 아니라 **「C4 가 못 잰다」**의 증거였다(R40 ②). "
             "★★★ 그래서 대조를 **사전확률 셔플**로 갈아끼운다 — 같은 기저·같은 시프트 값 "
             "집합·**구성으로 동일한 raw**, 유일한 차이는 「각 레코드가 자기 사전확률을 받는가」. "
             "이러면 **유병률 주입**과 **자기 사전확률 정렬**이 정확히 갈린다. "
             "★ 함께: Q3 의 실패 기전(π̂ 이 지배 레코드에서 clip 바닥으로 붕괴)을 특정하고, "
             "대안 추정기 BBSE 를 **|π̂−π*|** 로 비교한다(추정 지표라 pooled 오염과 무관)."),
    dataset="SVDB — svdb_data5.npz (리듬 특징만 · P 위치 자산도 BUT PDB 도 쓰지 않는다)",
    primary=PRIMARY, macro_role=MACRO_ROLE, arm_monotone=ARM_MONOTONE,
    read_order=READ_ORDER, gate_dep=GATE_DEP,
    n_boot=NB_BOOT, n_perm=N_PERM, n_shuf=N_SHUF, smoke=SMOKE,
    ref=REF, rule_check=RULE_CHECK,
    predictions={
        "B0": "재현·항등 — Q3 의 raw·매크로를 다시 내고(|Δ| ≤ 0.005), **셔플 팔의 raw 가 "
              "오라클 팔과 정확히 같음**을 구성으로 증명한다. 깨지면 **중단**",
        "B1": "★★★ **주 관문 — `oracle − shuffled` 짝지은 차.** 같은 부트스트랩 재표집에서 "
              "두 팔을 함께 재고 차를 낸다. CI 가 0 을 떼면 이득은 **자기 사전확률 정렬**에서 "
              "온다. 못 떼면 **유병률 주입 자체**가 이득의 정체이고, 그러면 전역 PR-AUC 위의 "
              "이득 해석을 **C2 를 포함해 폐기**한다",
        "B2": "★★ **C4 의 결함 교정** — 라벨치환 영점을 Δ 가 아니라 **절대 수준**으로 비교한다. "
              "Q3 이 안 남긴 영점 raw 를 **실측**해 기록한다",
        "B3": "EM 에도 같은 셔플 대조. EM 은 Q3 에서 이미 음수(Δ −0.0185)라 여기서 결론이 "
              "안 날 수 있다 — 그러면 그렇게 쓴다(미결 ≠ 등가)",
        "B4": "★★ **관문 아님 — 기전 특정.** π̂ 이 clip 바닥/천장에 붙은 레코드를 세고, 그 "
              "레코드들이 TEST S 에서 차지하는 **지분**을 찍는다. Q3 산점도의 (π* 0.576, π̂ 0.01) "
              "이 지배 레코드인지 확인한다",
        "B5": "**관문 아님 — 추정기 비교.** BBSE(혼동행렬 기반) · 평균사후 · EM 을 "
              "**|π̂−π*|** 로 비교한다. 이건 **추정 지표**라 pooled 오염과 무관하다. "
              "⚠️ MLLS 는 Saerens EM 과 **같은 것**이라 별도 팔이 아니다",
        "B6": "결론 검산표 — 판정마다 (a) 근거 (b) 미검정 가정 (c) 틀리면"},
    caveat=("★★★ **이 런은 새 가설을 시험하지 않는다 — 자를 고친다**(R35 ①). 그래서 B1 이 "
            "❌ 여도 그건 「방법 A 가 나쁘다」가 아니라 **「전역 PR-AUC 가 이 질문의 자가 아니다」**다. "
            "★ 셔플 팔은 **derangement** 다 — 자기 값을 받는 레코드가 하나도 없어야 대조가 "
            "희석되지 않는다. ★ 오라클은 여전히 **방법이 아니라 상한**이다(TEST 유병률 사용). "
            "★ Q3 과 **같은 분할·같은 기저·같은 보정 절차**를 밟는다 — 안 그러면 재현이 아니다.")
)
np.random.seed(SEED0)
run = MedKOSRun("quest46_q3b_prior_shuffle", CONFIG, project=PROJECT)
run.log("설정 ✅ **Q3-B — 사전확률 셔플 대조(Q3 의 C4 를 고친다)**")
run.log("  ★★★ 주 통계량 = **`oracle − shuffled` 짝지은 차** (같은 raw 위에서)")
run.log(f"  ★ Q3 실측 앵커 — raw {REF['raw']} · oracle {REF['oracle']} · em {REF['em']}")
run.log(f"  ★ C4 의 결함: 영점 raw ≈0.106 vs 관측 raw {REF['raw']} — **출발점이 달라 이득 비교 불가**")
if SMOKE:
    run.log(f"  ⚠️ **스모크런** — 비용 손잡이만 축소(NB_BOOT={NB_BOOT} · N_PERM={N_PERM} · "
            f"N_SHUF={N_SHUF}). 관문 문턱은 그대로다")
run.log("\n  사전등록 규칙 체크리스트 (R29 ③)")
for k_, v_ in RULE_CHECK.items():
    run.log(f"    [x] {k_:<18} {v_}")


In [ ]:
# CELL 2 — 【B-0】 자산 · 코호트 · 분할 · 기저 · DEV 전용 보정 (Q3 과 **같은 절차**)
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score, roc_auc_score

run.log("\n" + "=" * 100)
run.log("【B-0】 코호트 · 분할 · 기저 · DEV 전용 보정 — Q3 과 같은 결정 절차")
run.log("=" * 100)
VERD, DIFF = {}, {}
def g_(k, v, d):
    VERD[k] = v; run.log(f"  {k:<5}{v}  {d}")

if not os.path.exists(SV5):
    raise AssetError(f"{SV5} 없음 — svdb_labels.py build(R16)")
D5 = np.load(SV5, allow_pickle=True)
PID = np.asarray(D5["pid"]).astype(int)
Y3 = np.asarray(D5["y3"]).astype(int)
PRE = np.asarray(D5["pre_rr"], float); POST = np.asarray(D5["post_rr"], float)
K = np.where(Y3 >= 0)[0]
RID = PID[K]; Y = Y3[K]; TT = (Y == IDX_S)
pre = PRE[K].astype(float); post = POST[K].astype(float)
RS = np.array(sorted(set(RID.tolist())))

_S = pd.Series(pre); _G = _S.groupby(pd.Series(RID))
def local_base(k):
    r = np.asarray(_G.apply(lambda x: x.shift(1).rolling(k, min_periods=1).median())).astype(float)
    return np.where(np.isfinite(r), r, pre)
_med = _G.transform("median").to_numpy()
_std = _G.transform("std").to_numpy(); _mean = _G.transform("mean").to_numpy()
f1 = _med - pre
f2 = {k: 1.0 - pre / (local_base(k) + 1e-9) for k in FULL_K}
f3 = post - pre
f4 = np.nan_to_num(_std / (_mean + 1e-9))
RHY = np.c_[f1, np.column_stack([f2[k] for k in RHY_K]), f3, f4,
            np.log1p(pre), np.log1p(post)]
RHY = np.nan_to_num(RHY, nan=0.0, posinf=0.0, neginf=0.0)

IDXS_ALL = {r: np.where(RID == r)[0] for r in RS}
REC_OK = [r for r in RS
          if TT[IDXS_ALL[r]].sum() >= MIN_S and (~TT[IDXS_ALL[r]]).sum() >= MIN_N]
EXCL = [int(r) for r in RS if r not in REC_OK]
BURD_ALL = {int(r): float(TT[IDXS_ALL[r]].mean()) for r in RS}
run.log(f"  레코드 {len(RS)} · 채점 가능 {len(REC_OK)} · 제외 {len(EXCL)}")

order = sorted(REC_OK, key=lambda r: (BURD_ALL[int(r)], int(r)))
SPLIT = {int(r): SPLIT_PATTERN[i % len(SPLIT_PATTERN)] for i, r in enumerate(order)}
TR_R = [r for r in REC_OK if SPLIT[int(r)] == "TRAIN"]
DV_R = [r for r in REC_OK if SPLIT[int(r)] == "DEV"]
TE_R = [r for r in REC_OK if SPLIT[int(r)] == "TEST"]
if min(len(TR_R), len(DV_R), len(TE_R)) < 3:
    raise AssetError("분할이 너무 얕다")
sel = lambda rs: np.concatenate([IDXS_ALL[r] for r in rs])
TR_I, DV_I, TE_I = sel(TR_R), sel(DV_R), sel(TE_R)
SPREAD = {}
for nm, rr, ii in (("TRAIN", TR_R, TR_I), ("DEV", DV_R, DV_I), ("TEST", TE_R, TE_I)):
    bs = [BURD_ALL[int(r)] for r in rr]
    SPREAD[nm] = (float(min(bs)), float(max(bs)))
    run.log(f"  {nm:<6}레코드 {len(rr):>3} · 비트 {len(ii):>7,} · S 유병률 "
            f"{TT[ii].mean():.4f} (레코드별 {min(bs):.4f}~{max(bs):.4f})")
ALL_LO, ALL_HI = min(BURD_ALL[int(r)] for r in REC_OK), max(BURD_ALL[int(r)] for r in REC_OK)
cover = (SPREAD["TEST"][1] - SPREAD["TEST"][0]) / max(ALL_HI - ALL_LO, 1e-9)
if cover < 0.5:
    raise AssetError(f"분할이 유병률을 블록으로 갈랐다(커버리지 {cover:.2f})")

s_cnt = np.array([int(TT[IDXS_ALL[r]].sum()) for r in TE_R], float)
DOMINANT = float(s_cnt.max() / s_cnt.sum())
DOM_REC = int(TE_R[int(np.argmax(s_cnt))])
run.log(f"  ★ TEST 지배 지분 **{DOMINANT:.3f}** — 레코드 **{DOM_REC}** (유병률 "
        f"{BURD_ALL[DOM_REC]:.4f}) · 참고 Q3 {REF['dominant']}")
run.log("    ▸ **전역 수치를 단독 인용하지 않는다**(R11)")
# ★ DEV 가 TEST 의 유병률 극단을 덮는가 — Q3 의 clip 이 DEV 에서 골라졌으므로 중요하다
run.log(f"  ★★ DEV 유병률 범위 {SPREAD['DEV'][0]:.4f}~{SPREAD['DEV'][1]:.4f} vs "
        f"TEST {SPREAD['TEST'][0]:.4f}~{SPREAD['TEST'][1]:.4f}")
if SPREAD["TEST"][1] > SPREAD["DEV"][1]:
    run.log(f"     ⚠️ **TEST 최고 유병률({SPREAD['TEST'][1]:.4f})이 DEV 최고"
            f"({SPREAD['DEV'][1]:.4f})를 넘는다** — DEV 에서 고른 EM clip 이 그 극단을 "
            "못 덮는다. B4 에서 이게 π̂ 붕괴와 이어지는지 본다")

def fit_base(tr_idx, ytr):
    mu, sd = RHY[tr_idx].mean(0), RHY[tr_idx].std(0) + 1e-9
    lr = LogisticRegression(max_iter=3000, C=1.0)
    lr.fit((RHY[tr_idx] - mu) / sd, ytr)
    return lambda idx: lr.decision_function((RHY[idx] - mu) / sd)

score_of = fit_base(TR_I, TT[TR_I].astype(int))
SC_dev, SC_te = score_of(DV_I), score_of(TE_I)

half = len(DV_R) // 2
DV_FIT_R, DV_SEL_R = DV_R[:half], DV_R[half:]
in_ = lambda rs: np.isin(RID[DV_I], np.asarray(rs))
m_fit, m_sel = in_(DV_FIT_R), in_(DV_SEL_R)

def make_platt(s, y):
    lr = LogisticRegression(max_iter=3000, C=1e6)
    lr.fit(np.asarray(s).reshape(-1, 1), np.asarray(y).astype(int))
    return lambda v: lr.predict_proba(np.asarray(v).reshape(-1, 1))[:, 1]

def make_iso(s, y):
    ir = IsotonicRegression(out_of_bounds="clip", y_min=1e-6, y_max=1 - 1e-6)
    ir.fit(np.asarray(s), np.asarray(y).astype(float))
    return lambda v: np.clip(ir.predict(np.asarray(v)), 1e-6, 1 - 1e-6)

CAND = {}
for nm, mk in (("platt", make_platt), ("isotonic", make_iso)):
    f_ = mk(SC_dev[m_fit], TT[DV_I][m_fit])
    CAND[nm] = ece_of(f_(SC_dev[m_sel]), TT[DV_I][m_sel].astype(float))
CAL_NAME = min(CAND, key=lambda k: CAND[k])
ALT_NAME = [k for k in CAND if k != CAL_NAME][0]
MK = {"platt": make_platt, "isotonic": make_iso}
calib = MK[CAL_NAME](SC_dev, TT[DV_I])
calib_alt = MK[ALT_NAME](SC_dev, TT[DV_I])
EPS = 1e-6
P_dev = np.clip(calib(SC_dev), EPS, 1 - EPS)
P_te = np.clip(calib(SC_te), EPS, 1 - EPS)
P_te_alt = np.clip(calib_alt(SC_te), EPS, 1 - EPS)
PI_TR = float(TT[DV_I].mean())
ECE_DEV, ECE_TE = CAND[CAL_NAME], ece_of(P_te, TT[TE_I].astype(float))
run.log(f"  보정기 = **{CAL_NAME}** (DEV 반쪽 홀드아웃 ECE {ECE_DEV:.5f}) · TEST ECE {ECE_TE:.5f}")
# ★★ 사후확률의 **입자도(고유값 수)** — EM 은 이것에 민감하다.
#    isotonic 은 계단 함수라 고유값이 적고, 계단이 거칠면 EM 이 clip 경계로 달아난다.
#    (합성 실측: 계단 3개면 π̂ 이 clip 바닥, 같은 자료의 연속 사후확률이면 거의 정확)
GRAN, GRAN_ALT = int(len(np.unique(P_te))), int(len(np.unique(P_te_alt)))
run.log(f"  ★★ TEST 사후확률 **입자도** — {CAL_NAME} 고유값 **{GRAN}개** vs "
        f"{ALT_NAME} {GRAN_ALT}개")
run.log("     ▸ EM 은 계단이 거칠수록 경계로 달아난다 → B5 에서 두 보정기로 **민감도**를 본다")
run.log(f"  π_tr = **{PI_TR:.5f}** (DEV 유병률) · 참고 Q3 {REF['pi_tr']}")
CONFIG["cohort"] = dict(n_ok=len(REC_OK), excluded=EXCL, dominant=DOMINANT, dom_rec=DOM_REC,
                        n_train=len(TR_R), n_dev=len(DV_R), n_test=len(TE_R), pi_tr=PI_TR,
                        calibrator=CAL_NAME, alt_calibrator=ALT_NAME,
                        ece_dev=ECE_DEV, ece_test=ECE_TE, spread=SPREAD,
                        granularity=GRAN, granularity_alt=GRAN_ALT)
run.save_json("config", CONFIG)


In [ ]:
# CELL 3 — 【B-A】 ★ B0 — 재현 · 항등 · **셔플 팔의 raw 가 구성으로 동일**
run.log("\n" + "=" * 100)
run.log("【B-A】 B0 — Q3 재현 · 항등 · 셔플 대조의 **구성적 보장**")
run.log("=" * 100)

def logit(p):
    p = np.clip(np.asarray(p, float), 1e-12, 1 - 1e-12)
    return np.log(p) - np.log1p(-p)

def shift_logit(pi, pi_tr):
    """레코드 하나에 붙는 **상수** 로짓 시프트. π = π_tr 이면 정확히 0.0 을 낸다."""
    if pi == pi_tr:
        return 0.0
    return float(logit(pi) - logit(pi_tr))

L_raw = logit(P_te)
RID_te = RID[TE_I]; Y_te = TT[TE_I].astype(int)
TE_POS = {int(r): np.where(RID_te == r)[0] for r in TE_R}
PI_STAR = {int(r): float(Y_te[TE_POS[int(r)]].mean()) for r in TE_R}

def apply_arm(pi_by_rec):
    out = L_raw.copy()
    for r in TE_R:
        pos = TE_POS[int(r)]
        out[pos] = out[pos] + shift_logit(pi_by_rec[int(r)], PI_TR)
    return out

pooled_ap = lambda L: float(average_precision_score(Y_te, L))
def per_record(scores):
    ap, au = {}, {}
    for r in TE_R:
        pos = TE_POS[int(r)]; yy = Y_te[pos]
        if yy.sum() == 0 or yy.sum() == len(yy):
            continue
        ap[int(r)] = float(average_precision_score(yy, scores[pos]))
        au[int(r)] = float(roc_auc_score(yy, scores[pos]))
    return ap, au

L_ident = apply_arm({int(r): PI_TR for r in TE_R})
L_oracle = apply_arm(PI_STAR)
AP_RAW, AP_ORACLE = pooled_ap(L_raw), pooled_ap(L_oracle)
AP_raw_rec, AU_raw_rec = per_record(L_raw)
MACRO_RAW = float(np.mean(list(AP_raw_rec.values())))
MACRO_AU = float(np.mean(list(AU_raw_rec.values())))

# ── ★★★ 사전확률 셔플 팔 — derangement 로 π* 를 레코드끼리 바꾼다
TE_KEYS = [int(r) for r in TE_R]
SHUF_MAPS = []
for s_ in range(N_SHUF):
    rng_ = np.random.RandomState(SEED0 + 300 + s_)
    perm = derangement(len(TE_KEYS), rng_)
    SHUF_MAPS.append({TE_KEYS[i]: PI_STAR[TE_KEYS[perm[i]]] for i in range(len(TE_KEYS))})
L_shuf = [apply_arm(m) for m in SHUF_MAPS]
n_self = sum(sum(1 for k in TE_KEYS if m[k] == PI_STAR[k]) for m in SHUF_MAPS)

run.log(f"  재현 — raw {AP_RAW:.4f} (Q3 {REF['raw']}) · oracle {AP_ORACLE:.4f} (Q3 {REF['oracle']})")
run.log(f"         매크로 PR-AUC {MACRO_RAW:.6f} (Q3 {REF['macro_prauc']}) · "
        f"AUROC {MACRO_AU:.6f} (Q3 {REF['macro_auroc']})")
d_rep = max(abs(AP_RAW - REF["raw"]), abs(AP_ORACLE - REF["oracle"]))

# ★★ 재현 앵커는 **같은 코호트일 때만** 적용된다. 스모크 플래그로 관문을 끄지 않는다 —
#    끄고 켜는 스위치를 두면 공식 실행에서도 꺼질 수 있다. 대신 **자산이 그 코호트인지**로
#    판단한다(구성적 조건). 코호트가 다르면 재현은 **정의상 불가능**하므로 앵커를 적용하지
#    않고, 그 실행은 **공식 재현이 아니라 리허설**이라고 스스로 선언한다.
COHORT_MATCH = (len(RS) == 78 and len(REC_OK) == 56
                and len(TE_R) == REF["n_test"] and len(DV_R) == REF["n_dev"]
                and len(TR_R) == REF["n_train"])
CONFIG["cohort_match"] = bool(COHORT_MATCH)
if COHORT_MATCH:
    run.log(f"  ★ 코호트가 Q3 과 같다(78/56 · {REF['n_train']}/{REF['n_dev']}/{REF['n_test']}) "
            f"→ **재현 앵커를 적용한다** (허용 {TOL_REPRO})")
    if d_rep > TOL_REPRO:
        raise AssetError(f"B0 재현 실패(|Δ| {d_rep:.4f} > {TOL_REPRO}) — Q3 과 같은 계산이 "
                         "아니면 대조를 얹을 수 없다")
else:
    run.log(f"  ⚠️⚠️ **코호트가 Q3 과 다르다** (레코드 {len(RS)}/{len(REC_OK)} · "
            f"{len(TR_R)}/{len(DV_R)}/{len(TE_R)}) → 재현 앵커를 **적용하지 않는다**.")
    run.log("     ★ 이 실행은 **공식 재현이 아니라 리허설**이다 — B1 의 수치를 인용하지 마라.")
    run.log("       (대조의 **구성적 보장**은 코호트와 무관하게 아래에서 그대로 검사한다)")

# ★★★ 이 런의 핵심 보장: 셔플 팔의 raw 는 오라클 팔의 raw 와 **정확히 같다**
#    (같은 기저 점수를 쓰고 시프트만 다르므로 — 구성으로 참이고 런타임에 검사한다)
d_ident = float(np.max(np.abs(L_ident - L_raw)))
run.log(f"\n  B0 — 항등 팔 max|Δscore| = **{d_ident:.1e}** (정확히 0 이어야 한다)")
run.log(f"  B0 — 셔플 {N_SHUF}개 모두 **derangement** (자기 값을 받은 레코드 {n_self}개)")
run.log(f"  B0 — ★★★ 모든 팔이 **같은 기저 점수**에서 출발한다 → raw 가 구성으로 동일 "
        f"({AP_RAW:.4f})")
run.log("       ▸ **이것이 C4 가 못 한 것이다** — 라벨치환 영점은 raw 자체를 바꿔 버렸다")
if d_ident != 0.0 or n_self != 0:
    raise AssetError(f"B0 실패 — 항등 {d_ident:.1e} · 자기값 {n_self}개. "
                     "대조는 가정이 아니라 **구성**이어야 한다(R34 ③ · R35 ④)")
# 매크로 항등(단조 팔) — 여기서도 판정 지표가 아니라 검산이다
worst = 0.0
for nm, LL in [("identity", L_ident), ("oracle", L_oracle)] + \
              [(f"shuf{i}", L_shuf[i]) for i in range(N_SHUF)]:
    ap_, au_ = per_record(LL)
    worst = max(worst, max(abs(ap_[k] - AP_raw_rec[k]) for k in AP_raw_rec),
                max(abs(au_[k] - AU_raw_rec[k]) for k in AU_raw_rec))
run.log(f"  B0 — 매크로 max|Δ| = **{worst:.1e}** (전 팔 단조 → 항등 · 허용 {TOL_IDENT:.0e})")
if worst >= TOL_IDENT:
    raise AssetError(f"B0 실패 — 매크로가 움직였다({worst:.3e})")
g_("B0", "✅ 지지" if COHORT_MATCH else "⚠️ 리허설",
   (f"Q3 재현(|Δ| {d_rep:.4f}) · 항등 정확 · **셔플 팔이 오라클과 같은 raw 에서 출발한다**"
    if COHORT_MATCH else
    "구성적 보장(항등·derangement·동일 raw)은 섰지만 **코호트가 달라 재현 앵커가 없다** — "
    "리허설이므로 수치를 인용하지 않는다"))
CONFIG["B0"] = dict(ap_raw=AP_RAW, ap_oracle=AP_ORACLE, macro_prauc=MACRO_RAW,
                    macro_auroc=MACRO_AU, repro_delta=d_rep, ident=d_ident,
                    n_self=n_self, macro_delta=float(worst))
run.save_json("config", CONFIG)


In [ ]:
# CELL 4 — 【B-B】 ★★★ B1 — 주 관문: `oracle − shuffled` 짝지은 차
run.log("\n" + "=" * 100)
run.log("【B-B】 B1 — **주 관문**: 이득이 「자기 사전확률 정렬」에서 오는가")
run.log("=" * 100)
run.log("  ▸ oracle 과 shuffled 는 **같은 기저·같은 시프트 값 집합·같은 raw** 다.")
run.log("    유일한 차이는 **레코드↔사전확률 대응**이다. 그래서 차이가 나면 그건 대응의 몫이다")

def cluster_boot(arms, seed, nb):
    """★ 재표집 단위는 **레코드**. 모든 팔을 **같은 재표집**에 태워 짝지은 차를 만든다."""
    rng = np.random.RandomState(seed)
    recs = [int(r) for r in TE_R]
    out = {k: [] for k in arms}
    for _ in range(nb):
        pick = [recs[i] for i in rng.randint(0, len(recs), len(recs))]
        pos = np.concatenate([TE_POS[r] for r in pick])
        yy = Y_te[pos]
        if yy.sum() < 5 or yy.sum() == len(yy):
            continue
        for k, LL in arms.items():
            out[k].append(float(average_precision_score(yy, LL[pos])))
    return {k: np.asarray(v, float) for k, v in out.items()}

ARMS = {"raw": L_raw, "oracle": L_oracle}
for i in range(N_SHUF):
    ARMS[f"shuf{i}"] = L_shuf[i]
T0 = time.time()
B = cluster_boot(ARMS, SEED0 + 11, NB_BOOT)
SH = np.mean([B[f"shuf{i}"] for i in range(N_SHUF)], axis=0)   # 셔플 평균(같은 재표집)
AP_SHUF = float(np.mean([pooled_ap(L) for L in L_shuf]))
run.log(f"\n  ({time.time()-T0:.0f}초) 부트스트랩 {len(B['raw'])}회 × 팔 {len(ARMS)}개")
run.log(f"  전역 PR-AUC — raw **{AP_RAW:.4f}** · shuffled **{AP_SHUF:.4f}** · "
        f"oracle **{AP_ORACLE:.4f}**")
run.log(f"    (참고 매크로 {MACRO_RAW:.4f} — 오라클이 매크로 수준에 붙는지 보라)")

d_b1 = B["oracle"] - SH
B1_PT = AP_ORACLE - AP_SHUF
b1_lo, b1_hi = float(np.percentile(d_b1, 2.5)), float(np.percentile(d_b1, 97.5))
B1_MDE = mde(b1_lo, b1_hi)
run.log(f"\n  ★★★ B1 — **oracle − shuffled = {B1_PT:+.4f}** [{b1_lo:+.4f}, {b1_hi:+.4f}] · "
        f"MDE {B1_MDE:.4f}")
# 참고 — 셔플이 raw 대비 얼마나 올리나(= 유병률 주입 자체의 몫)
d_sh = SH - B["raw"]
sh_pt = AP_SHUF - AP_RAW
sh_lo, sh_hi = float(np.percentile(d_sh, 2.5)), float(np.percentile(d_sh, 97.5))
d_or = B["oracle"] - B["raw"]
or_pt = AP_ORACLE - AP_RAW
or_lo, or_hi = float(np.percentile(d_or, 2.5)), float(np.percentile(d_or, 97.5))
run.log(f"       성분 — oracle−raw {or_pt:+.4f} [{or_lo:+.4f}, {or_hi:+.4f}] (Q3 C2 {REF['c2_gain']:+.4f})")
run.log(f"              shuf −raw {sh_pt:+.4f} [{sh_lo:+.4f}, {sh_hi:+.4f}] "
        "← **유병률 주입 자체의 몫**")
run.log("       ▸ 차의 부호를 성분 없이 인용하지 않는다(R36 ⑤)")
b1_v = decide(b1_lo, b1_hi, 0.0, ">")
g_("B1", b1_v,
   "★★★ 이득은 **자기 사전확률 정렬**에서 온다 — 대응을 깨면 무너진다. "
   "C2 ✅ 가 **살아난다**" if b1_v.startswith("✅") else
   ("★★★ 대응을 깨도 이득이 그대로다 → 이득의 정체는 **유병률 주입**이고, "
    "**전역 PR-AUC 는 이 질문의 자가 아니다**. C2 를 포함해 전역 위의 이득 해석을 폐기한다"
    if b1_v.startswith("❌") else
    "★ 미결 — 대응의 몫을 이 표본에서 못 갈랐다. **등가가 아니다**(상한은 CI 상단)"))
CONFIG["B1"] = dict(ap_shuf=AP_SHUF, diff=B1_PT, lo=b1_lo, hi=b1_hi, mde=float(B1_MDE),
                    oracle_gain=dict(pt=or_pt, lo=or_lo, hi=or_hi),
                    shuf_gain=dict(pt=sh_pt, lo=sh_lo, hi=sh_hi), n_shuf=N_SHUF)
run.save_json("config", CONFIG)


In [ ]:
# CELL 5 — 【B-C】 ★★ B2 — 라벨치환 영점을 **절대 수준**으로 (C4 결함 교정) · B3 EM 셔플
run.log("\n" + "=" * 100)
run.log("【B-C】 B2 — 영점을 **Δ 가 아니라 절대 수준**으로 · B3 — EM 셔플 대조")
run.log("=" * 100)
run.log("  ▸ Q3 의 C4 는 영점 **이득**을 관측 **이득**과 비교했다. 두 이득은 **다른 raw** 에서")
run.log("    출발하므로 비교할 수 없다(천장 효과). 여기서는 **수준**을 비교하고, Q3 이 안 남긴")
run.log("    **영점 raw 를 실측**해 기록한다")

def em_prior(p, pi_tr, iters, tol, clip):
    pi = float(pi_tr)
    for _ in range(int(iters)):
        w = pi / pi_tr; v = (1.0 - pi) / (1.0 - pi_tr)
        num = w * p
        pp = num / (num + v * (1.0 - p))
        new = float(np.clip(pp.mean(), clip, 1.0 - clip))
        if abs(new - pi) < tol:
            pi = new; break
        pi = new
    return pi

DV_POS = {int(r): np.where(RID[DV_I] == r)[0] for r in DV_R}
PI_STAR_DEV = {int(r): float(TT[DV_I][DV_POS[int(r)]].mean()) for r in DV_R}
GRID = [(it, tl, cp) for it in (100, 500) for tl in (1e-6, 1e-9)
        for cp in (1e-4, 1e-3, 1e-2, 5e-2)]
best, BEST_HP = None, None
for hp in GRID:
    err = np.mean([abs(em_prior(P_dev[DV_POS[int(r)]], PI_TR, *hp) - PI_STAR_DEV[int(r)])
                   for r in DV_R])
    if best is None or err < best:
        best, BEST_HP = err, hp
EM_ITERS, EM_TOL, PI_CLIP = BEST_HP
run.log(f"\n  EM 하이퍼(DEV 고정) — iters={EM_ITERS} · tol={EM_TOL:g} · clip={PI_CLIP:g} · "
        f"DEV |π̂−π*| {best:.5f} (Q3 {REF['dev_pi_err']})")

# ── B2 — 라벨치환 영점. ★ **절대 수준**을 남긴다
T1 = time.time()
NUL = {"raw": [], "oracle": [], "em": []}
for s_ in range(N_PERM):
    rr = np.random.RandomState(SEED0 + 400 + s_)
    y_perm = TT[TR_I].astype(int)[rr.permutation(len(TR_I))]
    sc_ = fit_base(TR_I, y_perm)
    pd_, pt_ = sc_(DV_I), sc_(TE_I)
    cal_ = {"platt": make_platt, "isotonic": make_iso}[CAL_NAME](pd_, TT[DV_I])
    q_te = np.clip(cal_(pt_), EPS, 1 - EPS)
    pi_tr_n = float(TT[DV_I].mean())
    l0_ = logit(q_te)
    def arm_(pi_by):
        o = l0_.copy()
        for r in TE_R:
            p_ = TE_POS[int(r)]
            o[p_] = o[p_] + shift_logit(pi_by[int(r)], pi_tr_n)
        return o
    pih = {int(r): em_prior(q_te[TE_POS[int(r)]], pi_tr_n, EM_ITERS, EM_TOL, PI_CLIP)
           for r in TE_R}
    NUL["raw"].append(pooled_ap(l0_))
    NUL["oracle"].append(pooled_ap(arm_(PI_STAR)))
    NUL["em"].append(pooled_ap(arm_(pih)))
nr_m, nr_lo, nr_hi, _ = boot_mean(NUL["raw"], SEED0 + 61)
no_m, no_lo, no_hi, _ = boot_mean(NUL["oracle"], SEED0 + 62)
ne_m, ne_lo, ne_hi, _ = boot_mean(NUL["em"], SEED0 + 63)
run.log(f"\n  ({time.time()-T1:.0f}초) B2 — 영점(라벨치환) **절대 수준** reps={N_PERM}")
run.log(f"    영점 raw    **{nr_m:.4f}** [{nr_lo:.4f}, {nr_hi:.4f}]   ← ★ Q3 이 안 남긴 수")
run.log(f"    영점 oracle **{no_m:.4f}** [{no_lo:.4f}, {no_hi:.4f}]")
run.log(f"    관측 raw     {AP_RAW:.4f}          관측 oracle **{AP_ORACLE:.4f}**")
run.log(f"    ▸ 영점 이득 {no_m - nr_m:+.4f} vs 관측 이득 {or_pt:+.4f} — "
        f"**출발점이 {AP_RAW - nr_m:+.4f} 다르므로 이 둘을 직접 비교하지 않는다**")
b2_v = decide(no_lo, no_hi, AP_ORACLE, "<")
g_("B2", b2_v,
   f"관측 oracle 수준 {AP_ORACLE:.4f} 이 영점 oracle 수준 {no_m:.4f} 을 **넘는다** — "
   "이득이 기저의 판별력 위에 얹혀 있다" if b2_v.startswith("✅") else
   f"관측 oracle {AP_ORACLE:.4f} 이 영점 {no_m:.4f} 을 못 넘는다 — 전역 수준이 "
   "**기저와 무관**하게 결정된다")

# ── B3 — EM 팔에도 같은 셔플 대조
PI_HAT = {int(r): em_prior(P_te[TE_POS[int(r)]], PI_TR, EM_ITERS, EM_TOL, PI_CLIP)
          for r in TE_R}
L_em = apply_arm(PI_HAT)
AP_EM = pooled_ap(L_em)
EM_SHUF_MAPS = []
for s_ in range(N_SHUF):
    rng_ = np.random.RandomState(SEED0 + 500 + s_)
    perm = derangement(len(TE_KEYS), rng_)
    EM_SHUF_MAPS.append({TE_KEYS[i]: PI_HAT[TE_KEYS[perm[i]]] for i in range(len(TE_KEYS))})
L_em_shuf = [apply_arm(m) for m in EM_SHUF_MAPS]
A2 = {"em": L_em}
for i in range(N_SHUF):
    A2[f"emshuf{i}"] = L_em_shuf[i]
B2b = cluster_boot(A2, SEED0 + 11, NB_BOOT)
EMSH = np.mean([B2b[f"emshuf{i}"] for i in range(N_SHUF)], axis=0)
AP_EMSHUF = float(np.mean([pooled_ap(L) for L in L_em_shuf]))
d_b3 = B2b["em"] - EMSH
B3_PT = AP_EM - AP_EMSHUF
b3_lo, b3_hi = float(np.percentile(d_b3, 2.5)), float(np.percentile(d_b3, 97.5))
run.log(f"\n  B3 — EM {AP_EM:.4f} (Q3 {REF['em']}) · EM-shuffled {AP_EMSHUF:.4f}")
run.log(f"       `em − em_shuffled` = **{B3_PT:+.4f}** [{b3_lo:+.4f}, {b3_hi:+.4f}] · "
        f"MDE {mde(b3_lo, b3_hi):.4f}")
b3_v = decide(b3_lo, b3_hi, 0.0, ">")
g_("B3", b3_v,
   "EM 의 π̂ 도 **대응이 맞아서** 이득을 낸다" if b3_v.startswith("✅") else
   ("EM 의 π̂ 는 대응이 맞아도 이득이 없다 — 추정이 신호를 못 담았다"
    if b3_v.startswith("❌") else
    "미결 — EM 은 Q3 에서 이미 음수였고 여기서도 못 갈랐다. **등가가 아니다**"))
CONFIG["B2"] = dict(null_raw=dict(mean=nr_m, lo=nr_lo, hi=nr_hi),
                    null_oracle=dict(mean=no_m, lo=no_lo, hi=no_hi),
                    null_em=dict(mean=ne_m, lo=ne_lo, hi=ne_hi),
                    obs_raw=AP_RAW, obs_oracle=AP_ORACLE, n_perm=N_PERM)
CONFIG["B3"] = dict(ap_em=AP_EM, ap_em_shuf=AP_EMSHUF, diff=B3_PT, lo=b3_lo, hi=b3_hi,
                    em_iters=EM_ITERS, em_tol=EM_TOL, pi_clip=PI_CLIP, dev_pi_err=float(best))
run.save_json("config", CONFIG)


In [ ]:
# CELL 6 — 【B-D】 ★★ B4 π̂ 붕괴 진단 · B5 대안 추정기 · B6 검산표
run.log("\n" + "=" * 100)
run.log("【B-D】 B4 — π̂ 붕괴 기전 · B5 — 추정기 비교 · B6 — 결론 검산표")
run.log("=" * 100)

# ── B4 — 관문 아님. Q3 의 실패 기전을 **특정**한다
S_TOT = float(sum(int(Y_te[TE_POS[int(r)]].sum()) for r in TE_R))
rows = []
for r in TE_R:
    k = int(r); pos = TE_POS[k]
    rows.append(dict(rec=k, pi_star=PI_STAR[k], pi_hat=PI_HAT[k],
                     err=abs(PI_HAT[k] - PI_STAR[k]),
                     s_share=float(Y_te[pos].sum()) / S_TOT,
                     at_floor=bool(abs(PI_HAT[k] - PI_CLIP) < 1e-9),
                     at_ceil=bool(abs(PI_HAT[k] - (1 - PI_CLIP)) < 1e-9)))
rows.sort(key=lambda d: -d["s_share"])
FLOOR = [d for d in rows if d["at_floor"]]
run.log(f"  B4 — π̂ 이 clip 바닥({PI_CLIP:g})에 붙은 레코드 **{len(FLOOR)}개** · "
        f"천장 {sum(1 for d in rows if d['at_ceil'])}개")
run.log(f"       그 레코드들의 TEST S 지분 합 **{sum(d['s_share'] for d in FLOOR):.3f}**")
run.log(f"\n  {'rec':>5}{'π*':>9}{'π̂':>9}{'|err|':>9}{'S지분':>8}  비고")
for d in rows[:8]:
    tag = "★ **지배 레코드**" if d["rec"] == DOM_REC else ""
    if d["at_floor"]:
        tag += " ⚠️ clip 바닥"
    run.log(f"  {d['rec']:>5}{d['pi_star']:>9.4f}{d['pi_hat']:>9.4f}{d['err']:>9.4f}"
            f"{d['s_share']:>8.3f}  {tag}")
dom = next(d for d in rows if d["rec"] == DOM_REC)
run.log(f"\n  ★★ 지배 레코드 {DOM_REC} — π* {dom['pi_star']:.4f} · π̂ {dom['pi_hat']:.4f} · "
        f"S 지분 {dom['s_share']:.3f}")
if dom["at_floor"] or dom["err"] > 0.2:
    run.log(f"     ⚠️⚠️ **π̂ 이 π_tr({PI_TR:.4f})보다 아래로 갔다** → 이 레코드의 S 를 pooled "
            "**최하위로 밀어낸다**. Q3 의 ΔPR-AUC 가 음수인 이유가 여기다")
    run.log(f"     ▸ DEV 최고 유병률 {SPREAD['DEV'][1]:.4f} < 이 레코드 {dom['pi_star']:.4f} — "
            "**DEV 에서 고른 clip 이 이 극단을 못 덮었다**")
err_all = np.array([d["err"] for d in rows])
rho_pi = spearman([d["pi_star"] for d in rows], [d["pi_hat"] for d in rows])
run.log(f"  ★ 사후확률 입자도 — {CAL_NAME} 고유값 {GRAN}개 (대안 {ALT_NAME} {GRAN_ALT}개). "
        "계단이 거칠면 EM 이 경계로 달아난다 → B5 의 `em@" + ALT_NAME + "` 가 그 민감도다")
run.log(f"  |π̂−π*| 평균 {err_all.mean():.5f} · 중앙 {np.median(err_all):.5f} · "
        f"최대 {err_all.max():.5f} · ρ {rho_pi:+.4f}  (Q3 {REF['pi_err_mean']}/"
        f"{REF['pi_err_med']}/{REF['pi_err_max']} · ρ {REF['rho_pi']})")

# ── B5 — 추정기 비교. ★ **|π̂−π*|** 로 재므로 pooled 오염과 무관하다
#    ⚠️ MLLS 는 Saerens EM 과 같은 것이라 별도 팔이 아니다
th_grid = np.quantile(P_dev, np.linspace(0.5, 0.999, 60))
yd = TT[DV_I].astype(int)
J = [( (P_dev[yd == 1] > t).mean() - (P_dev[yd == 0] > t).mean(), t) for t in th_grid]
TH = float(max(J)[1])                       # ★ 문턱도 **DEV 에서** 고정
TPR = float((P_dev[yd == 1] > TH).mean()); FPR = float((P_dev[yd == 0] > TH).mean())
run.log(f"\n  B5 — BBSE 문턱(DEV Youden) {TH:.5f} · TPR {TPR:.4f} · FPR {FPR:.4f}")

def est_bbse(p):
    if TPR - FPR < 1e-6:
        return float("nan")
    return float(np.clip(((p > TH).mean() - FPR) / (TPR - FPR), PI_CLIP, 1 - PI_CLIP))
EST = {
    "em (=MLLS)": {int(r): PI_HAT[int(r)] for r in TE_R},
    "평균사후":   {int(r): float(np.clip(P_te[TE_POS[int(r)]].mean(), PI_CLIP, 1 - PI_CLIP))
                   for r in TE_R},
    "bbse":       {int(r): est_bbse(P_te[TE_POS[int(r)]]) for r in TE_R},
    # ★★ 민감도 — **선택되지 않은 보정기**의 사후확률로 같은 EM 을 돌린다.
    #    보정기는 DEV 홀드아웃 ECE 로 골랐는데, ECE 가 낮은 게 곧 **EM 에 좋은** 건 아니다:
    #    isotonic 은 계단 함수라 입자도가 낮고, EM 은 거기서 경계로 달아난다.
    #    ⚠️ 이건 **진단**이지 주 팔 교체가 아니다 — 채택하려면 다음 런에서 사전등록한다
    f"em@{ALT_NAME}": {int(r): em_prior(P_te_alt[TE_POS[int(r)]], PI_TR,
                                        EM_ITERS, EM_TOL, PI_CLIP) for r in TE_R},
}
run.log(f"  {'추정기':<12}{'평균|err|':>11}{'중앙':>9}{'최대':>9}{'ρ':>9}  지배레코드 π̂")
B5 = {}
for nm, d_ in EST.items():
    e_ = np.array([abs(d_[int(r)] - PI_STAR[int(r)]) for r in TE_R])
    m_, lo_, hi_, _ = boot_mean(e_, SEED0 + 81)
    rho_ = spearman([PI_STAR[int(r)] for r in TE_R], [d_[int(r)] for r in TE_R])
    B5[nm] = dict(mean=m_, lo=lo_, hi=hi_, med=float(np.median(e_)), max=float(e_.max()),
                  rho=float(rho_), dom=float(d_[DOM_REC]))
    run.log(f"  {nm:<12}{m_:>11.5f}{np.median(e_):>9.5f}{e_.max():>9.5f}{rho_:>+9.4f}"
            f"  {d_[DOM_REC]:.4f} (π* {PI_STAR[DOM_REC]:.4f})")
BEST_EST = min(B5, key=lambda k: B5[k]["mean"])
run.log(f"  ★ |π̂−π*| 최소 = **{BEST_EST}** — 관문이 아니라 **다음 런의 후보 지목**이다")
run.log("    ▸ 이 비교는 **추정 지표** 위에 있어 전역 PR-AUC 의 오염과 무관하다")

# ── 필요표본 · B6 검산표
run.log(f"\n  필요표본 (**우월 프레임** · 단위 = TEST 레코드 · 현재 {len(TE_R)}개)")
NEED = {}
for nm, eff, half_ in (("B1 oracle−shuf", B1_PT, B1_MDE), ("B3 em−shuf", B3_PT, mde(b3_lo, b3_hi))):
    n5 = need_super(len(TE_R), half_, eff, False); n8 = need_super(len(TE_R), half_, eff, True)
    zero = abs(eff) < half_
    NEED[nm] = dict(effect=float(eff), half=float(half_), sup50=float(n5), sup80=float(n8),
                    uninterpretable=bool(zero))
    run.log(f"  {nm:<16}{eff:>+9.4f}{half_:>9.4f}{n5:>9.0f}{n8:>9.0f}  "
            + ("★ **효과 ≈ 0 이라 해석 불가**(R41 ②)" if zero else "읽을 수 있다"))

run.log("\n  ★ B6 — **결론 검산표**")
CHECK = [
    dict(claim=f"셔플 대조가 **구성으로 공정하다** (raw 동일 {AP_RAW:.4f})",
         num=f"모든 팔이 같은 기저 점수 · 항등 팔 max|Δ| {d_ident:.0e} · derangement 자기값 {n_self}개",
         assume="**없음** — 구성으로 보장되고 런타임에 검사한다(R34 ③)",
         iffalse="—  ★ 이것이 Q3 의 C4 가 못 한 것이다(라벨치환은 raw 를 바꿨다)"),
    dict(claim=f"B1 `oracle − shuffled` = {B1_PT:+.4f} [{b1_lo:+.4f}, {b1_hi:+.4f}]",
         num=f"성분 oracle−raw {or_pt:+.4f} · shuf−raw {sh_pt:+.4f} · MDE {B1_MDE:.4f}",
         assume="TEST 유병률 분포가 셔플로 **재배치 가능**할 만큼 다양하다는 것",
         iffalse="유병률이 다 비슷하면 셔플이 아무것도 안 바꿔 대조가 **무력**해진다 — "
                 f"실측 범위 {SPREAD['TEST'][0]:.4f}~{SPREAD['TEST'][1]:.4f} 로 충분히 벌어져 있다"),
    dict(claim=f"B2 영점 **수준** — raw {nr_m:.4f} → oracle {no_m:.4f} vs 관측 {AP_ORACLE:.4f}",
         num=f"reps={N_PERM} · Q3 이 안 남긴 영점 raw 를 실측했다",
         assume="**없음** — 라벨만 치환해 같은 절차를 밟는다",
         iffalse=f"★ Q3 의 C4 는 이 둘의 **이득**을 비교했다. 출발점이 {AP_RAW - nr_m:+.4f} "
                 "다르므로 그 비교는 성립하지 않았다(R40 ②)"),
    dict(claim=f"B4 π̂ 붕괴 — 지배 레코드 {DOM_REC} 에서 π̂ {dom['pi_hat']:.4f} vs π* {dom['pi_star']:.4f}",
         num=f"clip 바닥 레코드 {len(FLOOR)}개 · S 지분 합 {sum(d['s_share'] for d in FLOOR):.3f}",
         assume="clip 바닥이 **EM 발산**의 결과라는 것(수렴 실패 vs 진짜 최적해는 미검정)",
         iffalse="바닥이 진짜 우도 최적해라면 문제는 clip 이 아니라 **모형 오설정**이다 — "
                 "B5 의 BBSE 가 같은 레코드에서 어떻게 나오는지가 단서다"),
    dict(claim=f"B5 추정기 — |π̂−π*| 최소는 **{BEST_EST}**",
         num=" · ".join(f"{k} {v['mean']:.5f}" for k, v in B5.items()),
         assume="**없음** — 추정 지표라 pooled 오염과 무관하다",
         iffalse="—  ⚠️ 단 이건 **관문이 아니라 후보 지목**이다. 채택은 다음 런에서 사전등록"),
    dict(claim="매크로는 여기서도 **판정 지표가 아니다**",
         num=f"전 팔 단조 → B0 에서 max|Δ| {worst:.0e} < {TOL_IDENT:.0e}",
         assume="**없음** — 상수 로짓 시프트의 성질이다",
         iffalse="—"),
    dict(claim=f"전역이 주 지표인 예외 런 (지배 지분 {DOMINANT:.3f})",
         num=f"TEST 레코드 {len(TE_R)} · 제외 {len(EXCL)} · GMIN_S = S≥{MIN_S} & N≥{MIN_N}",
         assume="Q3 의 표적이 환자 간 비교 가능성이라 pooled 에서만 보인다는 것",
         iffalse=f"★ 지배 지분 {DOMINANT:.3f} 는 높다 — 한 레코드가 전역을 끌고 간다. "
                 "**전역 단독 인용 금지**(R11)"),
]
for i, c in enumerate(CHECK, 1):
    run.log(f"\n  [{i}] **{c['claim']}**")
    run.log(f"      근거   {c['num']}")
    run.log(f"      가정   {c['assume']}")
    run.log(f"      틀리면 {c['iffalse']}")
CONFIG["B4"] = dict(rows=rows, floor=len(FLOOR),
                    floor_share=float(sum(d["s_share"] for d in FLOOR)),
                    dom=dom, rho=float(rho_pi), err_mean=float(err_all.mean()),
                    err_med=float(np.median(err_all)), err_max=float(err_all.max()))
CONFIG["B5"] = dict(est=B5, best=BEST_EST, th=TH, tpr=TPR, fpr=FPR)
CONFIG["need"] = NEED; CONFIG["B6"] = CHECK
run.save_json("config", CONFIG)


In [ ]:
# CELL 7 — 【B-E】 그림 · 요약 · 마무리
# ⚠️ Colab 기본 matplotlib 에 **한글이 없어** 축·범례·제목은 ASCII 로만 쓴다.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.6))

# ① 팔별 **절대 수준** — 관측 vs 영점 (C4 가 못 한 비교)
labs = ["raw", "shuffled", "oracle", "EM"]
obs = [AP_RAW, AP_SHUF, AP_ORACLE, AP_EM]
ax[0].bar(np.arange(4) - 0.18, obs, width=0.36, color="tab:red", label="observed")
ax[0].bar(np.array([0, 2, 3]) + 0.18, [nr_m, no_m, ne_m], width=0.36,
          color="tab:gray", label="label-permutation null")
ax[0].axhline(MACRO_RAW, ls="--", color="tab:blue", lw=1.0,
              label=f"macro = {MACRO_RAW:.3f}")
ax[0].set_xticks(range(4)); ax[0].set_xticklabels(labs, fontsize=8)
ax[0].set_ylabel("pooled PR-AUC (absolute level)")
ax[0].set_title("B2 : levels, not gains (fixes Q3's C4)", fontsize=9)
ax[0].legend(fontsize=7); ax[0].grid(alpha=.3, axis="y")

# ② B1 주 관문 — 짝지은 차와 그 성분
names = ["oracle - shuffled\n(B1, primary)", "oracle - raw", "shuffled - raw"]
vals = [B1_PT, or_pt, sh_pt]
los = [B1_PT - b1_lo, or_pt - or_lo, sh_pt - sh_lo]
his = [b1_hi - B1_PT, or_hi - or_pt, sh_hi - sh_pt]
cols = ["tab:red", "tab:gray", "tab:gray"]
ax[1].errorbar(vals, np.arange(3), xerr=[los, his], fmt="o", capsize=5, ls="none",
               ecolor="k", mfc="w")
for i, c in enumerate(cols):
    ax[1].plot(vals[i], i, "o", color=c, ms=8)
ax[1].axvline(0, color="k", lw=.9)
ax[1].set_yticks(range(3)); ax[1].set_yticklabels(names, fontsize=8)
ax[1].set_xlabel("pooled dPR-AUC (paired, same resample)")
ax[1].set_title("B1 : does the MATCHING matter?", fontsize=9)
ax[1].grid(alpha=.3, axis="x")

# ③ B5 추정기 — π* vs π̂
mk = {"em (=MLLS)": "o", "평균사후": "s", "bbse": "^", f"em@{ALT_NAME}": "D"}
cl = {"em (=MLLS)": "tab:purple", "평균사후": "tab:orange", "bbse": "tab:green",
      f"em@{ALT_NAME}": "tab:brown"}
en = {"em (=MLLS)": "EM (=MLLS)", "평균사후": "mean posterior", "bbse": "BBSE",
      f"em@{ALT_NAME}": f"EM on {ALT_NAME}"}
xs = [PI_STAR[int(r)] for r in TE_R]
for nm, d_ in EST.items():
    ax[2].scatter(xs, [d_[int(r)] for r in TE_R], s=30, marker=mk[nm], color=cl[nm],
                  alpha=.8, label=f"{en[nm]} (|err| {B5[nm]['mean']:.3f})")
m_ = max(max(xs), 0.6) * 1.05
ax[2].plot([0, m_], [0, m_], "k--", lw=.9)
ax[2].axhline(PI_CLIP, ls=":", color="tab:red", lw=1.0, label=f"clip floor {PI_CLIP:g}")
ax[2].axhline(PI_TR, ls=":", color="tab:blue", lw=1.0, label=f"pi_tr {PI_TR:.3f}")
ax[2].set_xlabel("true record prevalence pi*"); ax[2].set_ylabel("estimated pi_hat")
ax[2].set_title("B5 : estimators (metric immune to pooled artifact)", fontsize=9)
ax[2].legend(fontsize=6); ax[2].grid(alpha=.3)
fig.tight_layout()
PNG = run.save_fig("q3b_prior_shuffle", fig)
plt.close(fig); display(Image(PNG))

run.log("\n" + "=" * 100)
run.log("요약")
run.log("=" * 100)
if not COHORT_MATCH:
    run.log("  ⚠️⚠️ **리허설 — 코호트가 Q3 과 다르다. 아래 수치를 인용하지 마라.**")
    run.log("     (대조의 구성적 보장만 확인한 실행이다)")
    run.log("")
ok_ = lambda k: VERD.get(k, "").startswith("✅")
no_ = lambda k: VERD.get(k, "").startswith("❌")
for g in READ_ORDER[:4]:
    run.log(f"  {g:<5}{VERD.get(g, '(관문 아님)')}")
run.log(f"  B4   (관문 아님) π̂ 붕괴 진단 · B5   (관문 아님) 추정기 비교")
run.log("")
run.log(f"  전역 PR-AUC — raw {AP_RAW:.4f} · shuffled {AP_SHUF:.4f} · oracle {AP_ORACLE:.4f} "
        f"· EM {AP_EM:.4f} · 매크로 {MACRO_RAW:.4f}")
run.log(f"  영점 **수준** — raw {nr_m:.4f} · oracle {no_m:.4f}   ← Q3 이 안 남긴 수")
run.log("")
if ok_("B1"):
    run.log("  ★★★ **B1 ✅ — 이득은 「자기 사전확률 정렬」에서 온다.**")
    run.log(f"     대응을 깨면 {AP_ORACLE:.4f} → {AP_SHUF:.4f} 로 무너진다"
            f"(차 {B1_PT:+.4f} [{b1_lo:+.4f}, {b1_hi:+.4f}])")
    run.log("     → **Q3 의 C2 ✅ 가 살아난다.** C4 ⚠️ 는 대조가 잘못 짜인 탓이었다(R40 ②)")
    run.log(f"     → 방법 A 의 병목은 **추정**이다. B5 가 지목한 후보 = **{BEST_EST}**")
    run.log(f"     → 다음 런: 그 추정기로 **C3 를 다시** 판정한다(사전등록 필요)")
elif no_("B1"):
    run.log("  ⛔⛔ **B1 ❌ — 대응을 깨도 이득이 그대로다.**")
    run.log("     이득의 정체는 **유병률 주입 자체**이고, 전역 PR-AUC 는 이 질문의 자가 아니다")
    run.log("     → **Q3 의 C2 ✅ 를 폐기한다.** 그리고 ★ **Q4 의 사전등록 기준")
    run.log("        (「Q3 대비 전역 PR-AUC > 0」)도 같이 무효다** — 지표부터 새로 세운다")
else:
    run.log("  ⚠️ **B1 미결 — 대응의 몫을 이 표본에서 못 갈랐다.**")
    run.log(f"     상한은 CI 상단 {b1_hi:+.4f} 이고, **미결은 등가가 아니다**(R33 ① · R36 ①)")
    run.log(f"     성분 — oracle−raw {or_pt:+.4f} · shuf−raw {sh_pt:+.4f} "
            "(둘을 함께 보지 않으면 차의 부호를 못 읽는다 · R36 ⑤)")
run.log("")
if dom["at_floor"] or dom["err"] > 0.2:
    run.log(f"  ★★ **Q3 의 EM 실패 기전이 특정됐다** — 지배 레코드 {DOM_REC}(TEST S 의 "
            f"{dom['s_share']:.1%})에서 π̂ {dom['pi_hat']:.4f} 이 π* {dom['pi_star']:.4f} 은커녕")
    run.log(f"     π_tr {PI_TR:.4f} 보다도 **아래**다 → 그 레코드의 S 를 pooled 최하위로 민다")
    run.log(f"     ▸ DEV 최고 유병률 {SPREAD['DEV'][1]:.4f} 이 이 레코드를 못 덮어 "
            "**clip 이 DEV 에서 잘못 골라졌다**")
run.log("")
run.log("  ▸ ★ **전역 단독 인용 금지** — 지배 지분 "
        f"{DOMINANT:.3f} · 제외 {len(EXCL)} · GMIN_S(S≥{MIN_S}, N≥{MIN_N}) 와 함께만(R11)")
run.log("  ▸ ★ **오라클은 방법이 아니라 상한**이다 — TEST 유병률을 쓰므로 배포 불가")
run.log("  ▸ ★ 이 런은 가설이 아니라 **자(대조)를 고쳤다**(R35 ①)")

run.finish({
    "exp_id": "quest46_q3b_prior_shuffle",
    "metric": "oracle_minus_shuffled",
    "value": float(B1_PT),
    "passed": bool(COHORT_MATCH and ok_("B0") and ok_("B1") and ok_("B2")),
    "cohort_match": bool(COHORT_MATCH),
    "summary": ("Q3 의 C4 를 고치는 대조 수리 런. 라벨치환 영점은 raw 를 바꿔 이득 비교가 "
                "불가능했으므로, π* 를 레코드끼리 셔플(derangement)해 **raw 가 구성으로 동일한** "
                "대조를 만들었다. 주 관문은 `oracle − shuffled` 짝지은 차이고, 영점은 Δ 가 "
                "아니라 **절대 수준**으로 비교한다. π̂ 붕괴 기전과 대안 추정기(BBSE)도 진단한다."),
    "verdicts": VERD, "diffs": DIFF, "rule_check": RULE_CHECK,
    "cohort": CONFIG.get("cohort", {}), "B0": CONFIG.get("B0", {}), "B1": CONFIG.get("B1", {}),
    "B2": CONFIG.get("B2", {}), "B3": CONFIG.get("B3", {}), "B4": CONFIG.get("B4", {}),
    "B5": CONFIG.get("B5", {}), "B6": CONFIG.get("B6", []), "need": CONFIG.get("need", {}),
    "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `python pipelines/ingest_run.py --results result.json "
        "--notebook notebooks/quest46_q3b_prior_shuffle.ipynb`")
